# Форматы визуализации в Folium

В предыдущем ноутбуке ([map_1](map_1.ipynb)) мы собрали одну цельную карту: подложка, тематические слои, стили, подсказки и элементы управления.

Теперь посмотрим на **разные способы визуализации данных** в Folium — своего рода «витрину возможностей». Каждый раздел самостоятелен: в нём создаётся отдельная небольшая карта, демонстрирующая один приём.

Мы разберём:

1. **Подложки** — несколько базовых карт с переключением;
2. **Маркеры и иконки** — `Marker`, `Icon`, `DivIcon`, всплывающие окна `Popup`;
3. **Легенду** для категориальных данных;
4. **Плотность точек** — агрегация по сетке с хороплетом, `MarkerCluster`, `HeatMap` на реальных данных о кафе;
5. **Другие приёмы** — `DualMap`, `AntPath`.

Работаем на тех же данных Василеостровского района, что и в map_1, плюс добавим слой кафе — мы заранее выгрузили их из OpenStreetMap и сохранили в ту же папку с данными.

> Документация: [python-visualization.github.io/folium](https://python-visualization.github.io/folium/) и раздел [Plugins](https://python-visualization.github.io/folium/latest/user_guide/plugins.html)


## 0. Импорт библиотек и данные


In [ ]:
import geopandas as gpd
import folium
from folium import plugins

Загружаем данные района. К четырём наборам из map_1 добавился файл `cafes.geojson` — кафе, заранее выгруженные из OpenStreetMap. Файлы можно взять по [ссылке](https://drive.google.com/drive/folders/1jHFo4foUXtTc6Hlik7R_GxGBYWAs8zOF?usp=share_link) и положить в папку `../data/spb_vas/`.


In [ ]:
area       = gpd.read_file("../data/spb_vas/area.geojson")
landuse    = gpd.read_file("../data/spb_vas/landuse.geojson")
stations   = gpd.read_file("../data/spb_vas/metro.geojson")
isochrones = gpd.read_file("../data/spb_vas/isochrones.geojson")
cafes      = gpd.read_file("../data/spb_vas/cafes.geojson")

Вычислим центр района — он пригодится как стартовая точка для всех карт.


In [ ]:
center = area.geometry.centroid.iloc[0]
center_lat, center_lon = center.y, center.x

print(center_lat, center_lon)

## 1. Подложки (базовые карты)

Подложка — это фоновое изображение карты. В map_1 мы задавали одну подложку через параметр `tiles`. Но можно добавить **несколько** подложек как отдельные слои `TileLayer` и переключаться между ними в панели управления.

Создадим карту без подложки (`tiles=None`) и добавим несколько вариантов вручную.


In [ ]:
m1 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles=None)

# Светлая и тёмная подложки CartoDB
folium.TileLayer("cartodbpositron", name="CartoDB Positron (светлая)").add_to(m1)
folium.TileLayer("cartodbdark_matter", name="CartoDB Dark (тёмная)").add_to(m1)

# Классический OpenStreetMap
folium.TileLayer("openstreetmap", name="OpenStreetMap").add_to(m1)

# Спутниковый снимок Esri — задаётся через шаблон XYZ-тайлов и обязательную атрибуцию
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri Satellite (спутник)",
).add_to(m1)

folium.LayerControl(collapsed=False).add_to(m1)
m1

Порядок добавления важен: последняя добавленная подложка отображается по умолчанию.


### Откуда брать подложки

- **Встроенные** названия можно передавать прямо в `tiles=`: `"OpenStreetMap"`, `"CartoDB positron"`, `"CartoDB dark_matter"`. Полный список — в [документации Folium](https://python-visualization.github.io/folium/latest/user_guide/raster_layers/tiles.html).
- **Другие** подложки подключаются по шаблону URL `{z}/{x}/{y}` с атрибуцией `attr` (как спутник Esri выше). Готовые шаблоны удобно брать в галерее [Leaflet Providers](https://leaflet-extras.github.io/leaflet-providers/preview/).

> Часть провайдеров требует API-ключ и обязательного указания атрибуции — проверяйте условия использования.


## 2. Маркеры, иконки и всплывающие окна

В map_1 для станций метро мы использовали `CircleMarker` (кружок). Folium предлагает и другие способы отобразить точки.


### 2.1. `folium.Marker` с иконкой

`folium.Marker` — это классическая «булавка». Её вид настраивается через `folium.Icon`:

- `color` — цвет булавки (из фиксированного набора: `red`, `blue`, `green`, `darkred`, `cadetblue` и т. д.);
- `icon` — пиктограмма из набора [Font Awesome](https://fontawesome.com/icons);
- `prefix="fa"` указывает, что иконка берётся именно из Font Awesome.


In [ ]:
m2 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

for _, row in stations.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        icon=folium.Icon(color="darkred", icon="train", prefix="fa"),
        tooltip=row["name"],
    ).add_to(m2)

m2

### 2.2. `DivIcon` — подписи прямо на карте

`DivIcon` позволяет вывести на карту произвольный HTML вместо картинки — например, текстовую подпись станции.


In [ ]:
m3 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

for _, row in stations.iterrows():
    # сам маркер-кружок
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color="#334155", fill=True, fill_color="#334155", fill_opacity=1,
    ).add_to(m3)

    # текстовая подпись рядом с ним
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        icon=folium.DivIcon(
            icon_anchor=(-8, 10),
            html=f'<div style="font-size:11px; color:#334155; font-weight:600; white-space:nowrap;">{row["name"]}</div>',
        ),
    ).add_to(m3)

m3

### 2.3. `Popup` — всплывающее окно по клику

`tooltip` появляется при наведении, а `Popup` открывается по **клику** и может содержать полноценный HTML: заголовки, таблицы, ссылки, картинки.


In [ ]:
m4 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

for _, row in stations.iterrows():
    html = f"<b>{row['name']}</b><br>Широта: {row.geometry.y:.4f}<br>Долгота: {row.geometry.x:.4f}"
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        icon=folium.Icon(color="blue", icon="info-sign"),
        popup=folium.Popup(html, max_width=250),
        tooltip="Нажмите, чтобы узнать больше",
    ).add_to(m4)

m4

## 3. Легенда для категорий

В map_1 мы раскрасили землепользование по категориям, но карта осталась без легенды — читателю приходится догадываться, что означает каждый цвет. Добавим легенду.

Сначала повторим стилизацию landuse (как в map_1), а затем добавим легенду отдельным HTML-элементом.


In [ ]:
landuse_groups = {
    "residential": "residential",
    "commercial": "commercial", "retail": "commercial",
    "industrial": "industrial", "construction": "industrial",
    "garages": "industrial", "brownfield": "industrial",
    "grass": "green", "recreation_ground": "green",
    "flowerbed": "green", "farmland": "green",
    "forest": "forest",
    "religious": "special", "cemetery": "special", "military": "special",
}

landuse_colors = {
    "residential": "#fdd49e",
    "commercial":  "#fc8d59",
    "industrial":  "#d7301f",
    "green":       "#78c679",
    "forest":      "#238443",
    "special":     "#756bb1",
    "other":       "#d9d9d9",
}

landuse_labels = {
    "residential": "Жилая",
    "commercial":  "Коммерческая",
    "industrial":  "Промышленная",
    "green":       "Озеленение",
    "forest":      "Лес",
    "special":     "Специальная",
    "other":       "Прочее",
}

def landuse_style(feature):
    cls = landuse_groups.get(feature["properties"].get("landuse"), "other")
    return {
        "fillColor": landuse_colors[cls],
        "color": "#ffffff", "weight": 0.4,
        "fillOpacity": 0.55, "opacity": 0.6,
    }

Легенду соберём как обычный HTML-блок и добавим на карту через `get_root().html.add_child(...)`. Это универсальный приём: так на карту можно поместить любой свой HTML-элемент (заголовок, подпись, легенду).


In [ ]:
m5 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

folium.GeoJson(landuse, style_function=landuse_style, name="Землепользование").add_to(m5)

# Собираем HTML легенды из словаря: цветной прямоугольник + подпись для каждой группы
items = ""
for key, label in landuse_labels.items():
    items += f'<div><span style="background:{landuse_colors[key]}; padding:0 8px; margin-right:6px;"></span>{label}</div>'

legend_html = f"""
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background: white; padding: 10px; border: 1px solid #ccc; font-size: 13px;">
    <b>Землепользование</b>
    {items}
</div>
"""

m5.get_root().html.add_child(folium.Element(legend_html))
m5

## 4. Плотность точек

Когда точек много, отдельные маркеры сливаются в кашу — по такой карте уже не понять, где объектов больше, а где меньше. Есть несколько приёмов показать именно **плотность**: агрегировать точки по ячейкам сетки и раскрасить их (хороплет), сгруппировать близкие точки в кластеры (`MarkerCluster`) или построить тепловую карту (`HeatMap`). Разберём все три на реальных точках — **кафе** (файл `cafes.geojson`, загруженный в разделе 0).

### 4.1. Агрегация по сетке и хороплет

Раскрасить объекты можно не только по категории, но и по **числовому показателю**. `folium.Choropleth` строит **хороплет** — карту, где территориальные единицы закрашены оттенками в зависимости от значения показателя. В русскоязычной литературе такую карту называют **картограммой** (подробнее про картограммы и классификацию данных — в ноутбуке [visualizations_2](../module_4/visualizations_2.ipynb) модуля 4). Это высокоуровневый инструмент: он сам разбивает значения на классы, подбирает цвета и рисует легенду. Показатель может быть любым; здесь им будет число точек в ячейке.

Частый сценарий для плотности — **агрегация точек по ячейкам сетки**: делим территорию на равные квадраты, считаем число объектов в каждом и раскрашиваем квадраты по этому числу. Построим сетку 300×300 м и посчитаем, сколько кафе попадает в каждую ячейку.


Сетку построим функцией `create_regular_grid` — такой же, как в модуле 5. Она принимает слой и размер ячейки в метрах, при необходимости сама переводит данные в метрическую систему координат (UTM) и возвращает `GeoDataFrame` с квадратами и столбцом `cell_id`.


In [ ]:
from shapely.geometry import box


def create_regular_grid(data, cell_size):
    # если координаты географические (в градусах) — переводим в метрическую систему координат UTM
    if data.crs.is_geographic:
        data = data.to_crs(data.estimate_utm_crs())

    # границы области
    minx, miny, maxx, maxy = data.total_bounds

    grid_cells = []
    cell_ids = []

    x = minx
    cell_id = 0
    while x < maxx:
        y = miny
        while y < maxy:
            grid_cells.append(box(x, y, x + cell_size, y + cell_size))
            cell_ids.append(cell_id)
            cell_id += 1
            y += cell_size
        x += cell_size

    grid = gpd.GeoDataFrame({"cell_id": cell_ids, "geometry": grid_cells}, crs=data.crs)
    return grid


grid = create_regular_grid(area, 300)   # ячейки 300×300 м
print("Ячеек в сетке:", len(grid))

Теперь пространственным соединением (`sjoin`) определим, в какую ячейку попадает каждое кафе, и посчитаем число кафе в каждой ячейке.


In [ ]:
# приводим кафе к той же системе координат, что и сетка
cafes_grid = cafes.to_crs(grid.crs)

joined = gpd.sjoin(cafes_grid, grid, how="inner", predicate="within")
counts = joined.groupby("cell_id").size()

grid["cafes"] = grid["cell_id"].map(counts).fillna(0).astype(int)
grid["cafes"].describe()

Осталось нанести сетку на карту. Оставим только ячейки, где есть хотя бы одно кафе, и раскрасим их по числу кафе. Данные и геометрию свяжем по `cell_id`.


In [ ]:
grid_nonzero = grid[grid["cafes"] > 0]

m7 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

folium.Choropleth(
    geo_data=grid_nonzero,
    data=grid_nonzero,
    columns=["cell_id", "cafes"],
    key_on="feature.properties.cell_id",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    line_color="white",
    legend_name="Число кафе в ячейке (300×300 м)",
).add_to(m7)

m7

### 4.2. `MarkerCluster`

Плагин `MarkerCluster` группирует близкие точки в кластеры с числом объектов. При приближении кластеры распадаются на отдельные маркеры — так на одной карте видно и общую картину скоплений, и конкретные кафе.


In [ ]:
m8 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

# Folium вставляет текст подсказки в JS-шаблон, обрамлённый обратными кавычками (`),
# поэтому символ ` в названии кафе (например, "Es`Pressa") ломает скрипт карты — и она не отображается.
# Заменяем обратные кавычки на обычные, чтобы подсказки не рвали JavaScript.
cafes["name"] = cafes["name"].str.replace("`", "'", regex=False)

cluster = plugins.MarkerCluster(name="Кафе").add_to(m8)
for _, row in cafes.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row["name"],
        icon=folium.Icon(color="orange", icon="coffee", prefix="fa"),
    ).add_to(cluster)

m8

### 4.3. `HeatMap`

Плагин `HeatMap` строит тепловую карту плотности: там, где точек больше, «горячее». Параметры `radius` и `blur` управляют размытием.


In [ ]:
# HeatMap принимает список координат [широта, долгота]
coords = [[geom.y, geom.x] for geom in cafes.geometry]

m9 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbdark_matter")

plugins.HeatMap(coords, radius=18, blur=15, name="Плотность кафе").add_to(m9)
m9

## 5. Другие приёмы


### 5.1. `DualMap` — две карты рядом

`plugins.DualMap` показывает две синхронизированные карты бок о бок — удобно для сравнения (например, разных подложек или слоёв «до/после»). Слои можно добавлять сразу на обе карты (`dual`) или только на левую (`dual.m1`) / правую (`dual.m2`).


In [ ]:
dual = plugins.DualMap(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron",
)

# граница района — на обе карты
folium.GeoJson(area, style_function=lambda f: {"color": "#334155", "weight": 2, "fill": False}
               ).add_to(dual)

# землепользование — только на левую
folium.GeoJson(landuse, style_function=landuse_style, name="Землепользование").add_to(dual.m1)

# станции метро — только на правую
for _, row in stations.iterrows():
    folium.CircleMarker([row.geometry.y, row.geometry.x], radius=5,
                        color="#d7301f", fill=True, fill_opacity=1).add_to(dual.m2)

dual

### 5.2. `AntPath` — анимированный маршрут

`plugins.AntPath` рисует линию с «бегущим» пунктиром — наглядно для маршрутов и потоков. Построим условный маршрут, соединив станции метро по порядку.


In [ ]:
route = [[row.geometry.y, row.geometry.x] for _, row in stations.iterrows()]

m11 = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

plugins.AntPath(route, color="#d7301f", weight=5, delay=800).add_to(m11)

# отметим сами станции
for lat, lon in route:
    folium.CircleMarker([lat, lon], radius=4, color="#334155",
                        fill=True, fill_opacity=1).add_to(m11)

m11

## Итог

Мы прошлись по разным форматам визуализации в Folium:

- **подложки** — несколько базовых карт с переключением;
- **маркеры** — `Marker` с иконками, `DivIcon`-подписи, `Popup` по клику;
- **легенда** для категориальных данных через свой HTML;
- **плотность точек** — агрегация по сетке с хороплетом, `MarkerCluster` и `HeatMap` на данных о кафе из OpenStreetMap;
- **другие приёмы** — `DualMap`, `AntPath`.

Любую из этих карт можно сохранить в HTML (`m.save("index.html")`) и опубликовать — как разобрано в ноутбуке [map_3](map_3.ipynb).
